# RAG evaluation — FinVerify

This notebook runs our **implementation** financial advisor: **Claude (`claude-sonnet-4-6`)** augmented with retrieval over an authoritative knowledge base (IRS publications, CFPB guidance, SEC investor.gov, SSA, HealthCare.gov) persisted in a **ChromaDB** vector store.

Pipeline per question:

1. Embed the question with `BAAI/bge-small-en-v1.5`.
2. Retrieve top-k chunks from Chroma (`k=5`), filterable by topic when useful.
3. Build a prompt that includes retrieved passages with inline citation markers.
4. Call Claude with instructions to cite sources and refuse to fabricate when evidence is thin.
5. Grade with the same three signals as the baseline (MC accuracy, embedding similarity, LLM-as-judge) plus a **citation coverage** signal.

## 1. Prerequisites

```bash
python src/knowledge_base/ingest.py    # builds src/knowledge_base/vector_store/
export ANTHROPIC_API_KEY=sk-ant-...     # for Claude generation and the judge
```

In [ ]:
import os, sys, json, time, re, platform, subprocess
from pathlib import Path

REPO_URL = "https://github.com/niksharma99/COMS6156FinalProject.git"
REPO_NAME = "COMS6156FinalProject"
# TODO: switch to "main" once the eval/demo branch is merged.
REPO_BRANCH = "add-evaluation-dataset"

IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB:
    clone_target = Path("/content") / REPO_NAME
    if not clone_target.exists():
        print(f"Colab detected - cloning {REPO_URL} ({REPO_BRANCH})")
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(clone_target)],
            check=True,
        )
    os.chdir(clone_target)
    REPO_ROOT = clone_target
else:
    REPO_ROOT = Path.cwd()
    while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "dataset").exists():
        REPO_ROOT = REPO_ROOT.parent
    if not (REPO_ROOT / "dataset").exists():
        raise RuntimeError(
            f"Could not find repo root (no 'dataset/' dir walking up from {Path.cwd()})."
        )

SRC_DIR = str(REPO_ROOT / "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

try:
    from dotenv import load_dotenv
    load_dotenv(REPO_ROOT / ".env")
except ImportError:
    print("python-dotenv not installed; relying on shell environment for API keys.")

from eval.sampling import sample_from_dataset
from eval.metrics import grade_mc, cosine_similarity, judge_with_claude

VECTOR_DIR = REPO_ROOT / "src" / "knowledge_base" / "vector_store"

# The vector store is gitignored (build artifact). Build it if missing.
needs_build = (not VECTOR_DIR.exists()) or not any(VECTOR_DIR.iterdir())
if needs_build:
    print("Vector store missing - running ingest.py (first run will download sources + embed)...")
    subprocess.run([sys.executable, str(REPO_ROOT / "src" / "knowledge_base" / "ingest.py")], check=True)

assert VECTOR_DIR.exists() and any(VECTOR_DIR.iterdir()), "Vector store build failed."
assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in .env or your shell."
print("Repo root:", REPO_ROOT)
print("Python:", platform.python_version())
print("In Colab:", IN_COLAB)


## 2. Build the evaluation set

Default is `MODE = "all"` (all 156 questions) so the run is apples-to-apples with the baseline. Flip to `"sample"` for a quick smoke test (uses the same `seed=7` the baseline uses in sample mode).


In [ ]:
# MODE = "all"     -> every question in every topic file (matches the baseline run)
# MODE = "sample"  -> N_PER_DATASET stratified per dataset (fast smoke test, same seed as baseline)
MODE = "all"
N_PER_DATASET = 5

DATASETS = ["standard_questions", "open_ended_hard", "reddit_questions"]
TOPICS = ["budgeting", "credit_and_debt", "insurance", "investing", "retirement", "tax"]

def load_all_from_dataset(name: str) -> list[dict]:
    base = REPO_ROOT / "dataset" / name
    out = []
    for t in TOPICS:
        for it in json.loads((base / f"{t}.json").read_text()):
            out.append({**it, "_dataset": name, "_topic": t})
    return out

items = []
if MODE == "all":
    for ds in DATASETS:
        items.extend(load_all_from_dataset(ds))
else:
    for ds in DATASETS:
        items.extend(sample_from_dataset(ds, N_PER_DATASET, seed=7))

from collections import Counter
print(f"MODE={MODE}  total={len(items)}")
print("  by dataset:", dict(Counter(i['_dataset'] for i in items)))
print("  by type:   ", dict(Counter('MC' if i.get('type')=='multiple_choice' else 'OE' for i in items)))


## 3. Connect to the vector store

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

chroma = chromadb.PersistentClient(path=str(VECTOR_DIR))
coll = chroma.get_collection("finverify_kb")
print(f"Collection size: {coll.count()} chunks")

embedder = SentenceTransformer("BAAI/bge-small-en-v1.5")


def retrieve(question: str, k: int = 5, topic: str | None = None) -> list[dict]:
    q_emb = embedder.encode([question], normalize_embeddings=True)[0].tolist()
    where = {"topic": topic} if topic else None
    res = coll.query(query_embeddings=[q_emb], n_results=k, where=where)
    hits = []
    for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
        hits.append({"text": doc, "meta": meta, "distance": dist})
    return hits


# Sanity check
for h in retrieve("What is the difference between a traditional IRA and a Roth IRA?"):
    print(f"  [{h['meta']['publisher']}] {h['meta']['title']}  (d={h['distance']:.3f})")
    print(f"    {h['text'][:120]}")

## 4. RAG prompt

Each retrieved chunk is assigned an index `[1]`, `[2]`, .... Claude is instructed to cite chunk indices inline for every substantive claim and to say so explicitly when the passages don't contain enough evidence. This gives us a way to measure *citation coverage* later.

In [ ]:
SYSTEM_RAG = (
    "You are FinVerify, a careful personal-finance assistant. "
    "Answer using ONLY the provided passages when they are relevant. "
    "Cite sources inline as bracketed indices like [1], [2] after each claim they support. "
    "If the passages are insufficient to answer confidently, say so rather than fabricating. "
    "Acknowledge tradeoffs when they exist."
)

def build_rag_messages(item: dict, hits: list[dict]) -> list[dict]:
    ctx_lines = []
    for i, h in enumerate(hits, 1):
        m = h["meta"]
        ctx_lines.append(f"[{i}] ({m['publisher']} — {m['title']})\n{h['text']}")
    context_block = "\n\n".join(ctx_lines)

    if item.get("type") == "multiple_choice":
        options = "\n".join(item["options"])
        user = (
            f"Context:\n{context_block}\n\n"
            f"Question: {item['question']}\n\n{options}\n\n"
            "Respond with ONLY the single letter (A, B, C, or D)."
        )
    else:
        user = (
            f"Context:\n{context_block}\n\n"
            f"Question: {item['question']}\n\n"
            "Answer in 4-8 sentences. Cite passages as [n] after every substantive claim."
        )
    return user

## 5. Generate with Claude + RAG

In [ ]:
# Resume-safe RAG run. Checkpoints to disk after each item; restarting the kernel
# re-uses completed answers via `id`.
import anthropic
from anthropic import APIStatusError, APIConnectionError, RateLimitError

client = anthropic.Anthropic()
MODEL = "claude-sonnet-4-6"
K = 5

RESULTS_DIR = REPO_ROOT / "src" / "notebooks" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_PATH = RESULTS_DIR / f"rag_{MODEL}_all.json"

if RESULTS_PATH.exists():
    results = json.loads(RESULTS_PATH.read_text())
    print(f"Resuming: loaded {len(results)} existing results from {RESULTS_PATH.name}")
else:
    results = []
done_ids = {r["id"] for r in results}

remaining = [it for it in items if it["id"] not in done_ids]
total = len(remaining)
print(f"Running RAG on {total} remaining item(s) of {len(items)} total\n", flush=True)

def call_claude_with_retry(msg, system, max_tokens=700, max_attempts=4):
    delay = 2.0
    for attempt in range(1, max_attempts + 1):
        try:
            return client.messages.create(
                model=MODEL, max_tokens=max_tokens,
                system=system,
                messages=[{"role": "user", "content": msg}],
            )
        except (RateLimitError, APIConnectionError, APIStatusError) as e:
            if attempt == max_attempts:
                raise
            print(f"    retry {attempt}/{max_attempts} after {type(e).__name__}: sleeping {delay:.1f}s", flush=True)
            time.sleep(delay)
            delay *= 2

t_run_start = time.time()
for i, item in enumerate(remaining, 1):
    t0 = time.time()
    hits = retrieve(item["question"], k=K, topic=item["_topic"])
    user_msg = build_rag_messages(item, hits)
    resp = call_claude_with_retry(user_msg, SYSTEM_RAG, max_tokens=700)
    answer = resp.content[0].text.strip()
    dt = time.time() - t0
    results.append({
        "id": item["id"], "dataset": item["_dataset"], "topic": item["_topic"],
        "type": item.get("type"), "difficulty": item.get("difficulty"),
        "question": item["question"],
        "reference": item["correct_answer"],
        "candidate": answer,
        "retrieved": [{"title": h["meta"]["title"], "publisher": h["meta"]["publisher"],
                       "source_slug": h["meta"].get("source_slug"), "distance": h["distance"]}
                      for h in hits],
        "latency_sec": round(dt, 2),
    })
    RESULTS_PATH.write_text(json.dumps(results, indent=2))
    n_cites = len(re.findall(r'\[\d+\]', answer))
    print(f"[{i:3d}/{total}] {item['_dataset']}/{item['_topic']}/{item['id']:<12}  {dt:5.1f}s  cites={n_cites}", flush=True)

dt_total = time.time() - t_run_start
print(f"\nDone. {len(results)}/{len(items)} items. This run: {dt_total:.1f}s "
      f"({dt_total/max(total,1):.1f}s/item)")
print(f"Saved to {RESULTS_PATH}")


## 6. Score

### 6a. MC accuracy

In [ ]:
mc_rows = [r for r in results if r["type"] == "multiple_choice"]
for r in mc_rows:
    g = grade_mc(r["candidate"], r["reference"])
    r["mc_correct"] = g["is_correct"]; r["mc_picked"] = g["picked"]
if mc_rows:
    correct = sum(1 for r in mc_rows if r["mc_correct"])
    print(f"MC accuracy: {correct}/{len(mc_rows)} = {correct/len(mc_rows):.1%}")

### 6b. Embedding similarity (open-ended)

In [ ]:
oe_rows = [r for r in results if r["type"] != "multiple_choice"]
for r in oe_rows:
    a, b = embedder.encode([r["reference"], r["candidate"]], normalize_embeddings=True)
    r["embed_cosine"] = cosine_similarity(a, b)
if oe_rows:
    import statistics
    print(f"Mean cosine vs reference: {statistics.mean(r['embed_cosine'] for r in oe_rows):.3f}")

### 6c. LLM-as-judge rubric

In [ ]:
# Resume-safe judge loop: skip OE rows that already have `judge` populated.
judge_client = anthropic.Anthropic()
to_judge = [r for r in oe_rows if "judge" not in r]
print(f"Judging {len(to_judge)} of {len(oe_rows)} OE rows (skipping already-judged)")

for r in to_judge:
    try:
        score = judge_with_claude(r["question"], r["reference"], r["candidate"], client=judge_client)
        r["judge"] = score.to_dict()
    except Exception as e:
        print(f"  {r['id']}: judge failed ({type(e).__name__}) - skipping")
        continue
    print(f"  {r['id']:<12}  f={score.factuality} c={score.completeness} a={score.advice_quality}  mean={score.mean:.2f}")

# Persist the enriched results so judge scores survive a kernel restart.
RESULTS_PATH.write_text(json.dumps(results, indent=2))


### 6d. Citation coverage

Counts how many of the retrieved passages the model actually cited. Low values mean the model is ignoring retrieval; high values mean it's using the grounded context.

In [ ]:
for r in results:
    cited = set(int(x) for x in re.findall(r"\[(\d+)\]", r["candidate"]))
    cited = {c for c in cited if 1 <= c <= len(r["retrieved"])}
    r["n_citations"] = len(cited)
    r["citation_coverage"] = len(cited) / max(len(r["retrieved"]), 1)
import statistics
print(f"Mean citation coverage: {statistics.mean(r['citation_coverage'] for r in results):.2f}")

## 7. Summary & side-by-side with the baseline

In [ ]:
import pandas as pd
df = pd.DataFrame(results)

def agg(g):
    row = {"n": len(g)}
    if "mc_correct" in g.columns:
        mc = g.dropna(subset=["mc_correct"])
        row["mc_accuracy"] = mc["mc_correct"].mean() if len(mc) else None
    if "embed_cosine" in g.columns:
        oe = g.dropna(subset=["embed_cosine"])
        if len(oe):
            row["mean_cosine"] = oe["embed_cosine"].mean()
    if "judge" in g.columns:
        judged = g.dropna(subset=["judge"])
        if len(judged):
            row["judge_mean"] = judged["judge"].apply(lambda j: j["mean"]).mean()
    if "citation_coverage" in g.columns:
        row["citation_coverage"] = g["citation_coverage"].mean()
    row["mean_latency_sec"] = g["latency_sec"].mean()
    return pd.Series(row)

rag_summary = df.groupby("dataset").apply(agg, include_groups=False)
print("=== RAG ===")
print(rag_summary)

# Align with the baseline (sharded layout: results/baseline_<model>/all.json).
base_dir = REPO_ROOT / "src" / "notebooks" / "results"
baseline_candidates = sorted(base_dir.glob("baseline_*/all.json")) or sorted(base_dir.glob("baseline_*.json"))
if baseline_candidates:
    bp = baseline_candidates[0]
    print(f"\nBaseline file: {bp.relative_to(REPO_ROOT)}")
    base = pd.DataFrame(json.loads(bp.read_text()))
    print("\n=== Baseline ===")
    print(base.groupby("dataset").apply(agg, include_groups=False))
else:
    print("\n(no baseline results on disk - skipping side-by-side)")


## 8. Persist results

In [ ]:
# Shard results the same way the baseline does so comparison tooling is symmetric.
from collections import defaultdict

MODEL_SLUG = MODEL  # already a plain slug
OUT_DIR = REPO_ROOT / "src" / "notebooks" / "results" / f"rag_{MODEL_SLUG}"
OUT_DIR.mkdir(parents=True, exist_ok=True)

(OUT_DIR / "all.json").write_text(json.dumps(results, indent=2))

by_dataset: dict[str, list[dict]] = defaultdict(list)
for r in results: by_dataset[r["dataset"]].append(r)
for ds, rows in by_dataset.items():
    (OUT_DIR / f"{ds}.json").write_text(json.dumps(rows, indent=2))

by_dt: dict[tuple[str, str], list[dict]] = defaultdict(list)
for r in results: by_dt[(r["dataset"], r["topic"])].append(r)
topic_dir = OUT_DIR / "by_topic"; topic_dir.mkdir(exist_ok=True)
for (ds, tp), rows in by_dt.items():
    (topic_dir / f"{ds}__{tp}.json").write_text(json.dumps(rows, indent=2))

by_type: dict[str, list[dict]] = defaultdict(list)
for r in results:
    key = "multiple_choice" if r.get("type") == "multiple_choice" else "open_ended"
    by_type[key].append(r)
for k, rows in by_type.items():
    (OUT_DIR / f"type_{k}.json").write_text(json.dumps(rows, indent=2))

print(f"Wrote {len(results)} results to {OUT_DIR}")
print(f"  datasets: {sorted(by_dataset)}")
print(f"  topic shards: {len(by_dt)}")
print(f"  types: {sorted(by_type)}")
